# Fig S5D — Pseudotime segments: ATAC vs RNA embedding
The 16 adaptive even-trajectory pseudotime segments (equal-width DPT steps of 0.02 merged forward to ≥2,000 cells — the segments used to re-call peaks in 09_PseudotimePeaks) shown on the **ATAC-profile UMAP (left)** and the **RNA UMAP (right)**. Segments are defined once from pseudotime and colored identically in both panels, so a concordant layout shows the trajectory bins occupy matching territories in chromatin and transcriptome.

In [ ]:
from paperfig_style import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
BIN_WIDTH, BIN_FLOOR = 0.02, 2000
# --- ATAC cells: ATAC-UMAP coords + RNA pseudotime mapped by shared barcode ---
atac = pd.read_csv(os.path.join(CSV_DIR, 'ATAC_UMAP.csv'))
atac['pt'] = atac['cellName'].map(load_obs_by_atac('dpt_pseudotime'))
atac = atac[atac['pt'].notna()].copy()
# --- define segments once from ATAC pseudotime (same scheme as 09_PseudotimePeaks) ---
pt = atac['pt'].values
maxb = int(np.floor(pt.max()/BIN_WIDTH))
b = np.minimum((pt//BIN_WIDTH).astype(int), maxb)
cnt = np.bincount(b, minlength=maxb+1)
newid = np.zeros(maxb+1, int); cur=1; acc=0
for i in range(maxb+1):
    newid[i]=cur; acc+=cnt[i]
    if acc>=BIN_FLOOR: cur+=1; acc=0
if acc>0 and cur>1: newid[newid==cur]=cur-1
atac['seg'] = newid[b]
nseg = int(atac['seg'].max())
# --- RNA cells: multiome UMAP + pseudotime, SAME pt->segment mapping ---
rna_umap, obs = load_umap_obs(('dpt_pseudotime',))
rna_pt = np.asarray(obs['dpt_pseudotime'], float)
valid = np.isfinite(rna_pt)
rb = np.minimum((rna_pt[valid]//BIN_WIDTH).astype(int), maxb)
rna_seg = newid[rb]; rna_u = rna_umap[valid]
print('segments:', nseg, '| ATAC cells:', len(atac), '| RNA cells:', int(valid.sum()))
cmap = plt.get_cmap('turbo', nseg)
fig, axs = plt.subplots(1, 2, figsize=(7.6, 3.4))
o = np.argsort(atac['seg'].values)
axs[0].scatter(atac['UMAP1'].values[o], atac['UMAP2'].values[o], c=atac['seg'].values[o],
               cmap=cmap, vmin=0.5, vmax=nseg+0.5, s=1.2, linewidths=0, rasterized=True)
axs[0].set_title(f'ATAC UMAP \u2014 {nseg} pseudotime segments', fontsize=8)
axs[0].set_xlabel('ATAC-UMAP1'); axs[0].set_ylabel('ATAC-UMAP2')
o2 = np.argsort(rna_seg)
sc = axs[1].scatter(rna_u[o2,0], rna_u[o2,1], c=rna_seg[o2],
                    cmap=cmap, vmin=0.5, vmax=nseg+0.5, s=1.2, linewidths=0, rasterized=True)
axs[1].set_title(f'RNA UMAP \u2014 {nseg} pseudotime segments', fontsize=8)
axs[1].set_xlabel('RNA-UMAP1'); axs[1].set_ylabel('RNA-UMAP2')
cb = fig.colorbar(sc, ax=axs, fraction=0.03, pad=0.02); cb.set_label('pseudotime segment (1 = NE root)')
cb.outline.set_linewidth(0.5)
for ax in axs:
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ['left','bottom']: ax.spines[sp].set_visible(False)
savepanel(fig, 'FigS5D_PseudotimeSegments_UMAP')
